<a href="https://colab.research.google.com/github/milicaclm/transportni_problem/blob/main/transportni_problem_projekat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
kapacitet = [2, 6, 7]
potražnja = [3, 3, 4, 5]

cene = np.array([
    [20, 11, 15, 13],
    [17, 14, 12, 13],
    [15, 12, 18, 18]
])

cost_df = pd.DataFrame(cene, columns=['D1', 'D2', 'D3', 'D4'], index=['F1', 'F2', 'F3'])

print("Kapaciteti (Supply):", kapacitet)
print("Zahtevi (Demand):", potražnja)
print("Troškovi transporta (Costs):")
print(cost_df)

Kapaciteti (Supply): [2, 6, 7]
Zahtevi (Demand): [3, 3, 4, 5]
Troškovi transporta (Costs):
    D1  D2  D3  D4
F1  20  11  15  13
F2  17  14  12  13
F3  15  12  18  18


In [3]:
def severozapadni_ugao(kapacitet, potražnja):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    i = 0
    j = 0

    while i < len(kapacitet) and j < len(potražnja):
        if kap[i] <= pot[j]:
            resenje[i][j] = kap[i]
            pot[j] -= kap[i]
            i += 1
        else:
            resenje[i][j] = pot[j]
            kap[i] -= pot[j]
            j += 1

    return resenje

In [4]:
def metod_najmanjih_cena(kapacitet, potražnja, cene):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    cost_indices = [(cene[i][j], i, j) for i in range(len(kapacitet)) for j in range(len(potražnja))]
    cost_indices.sort()

    for cost, i, j in cost_indices:
        if kap[i] == 0 or pot[j] == 0:
            continue
        allocation = min(kap[i], pot[j])
        resenje[i][j] = allocation
        kap[i] -= allocation
        pot[j] -= allocation

    return resenje

In [5]:
def fogelov_metod(kapacitet, potražnja, costs):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    INF = int(1e10)

    while np.sum(kap) > 0 and np.sum(pot) > 0:

        redovni_penali = []
        for i in range(len(kap)):
            if kap[i] == 0:
                redovni_penali.append(-INF)
            else:
                red = sorted([costs[i][j] for j in range(len(pot)) if pot[j] > 0])
                if len(red) > 1:
                    redovni_penali.append(red[1] - red[0])
                else:
                    redovni_penali.append(red[0] if len(red) > 0 else -INF)

        kolonski_penali = []
        for j in range(len(pot)):
            if pot[j] == 0:
                kolonski_penali.append(-INF)
            else:
                kolona = sorted([costs[i][j] for i in range(len(kap)) if kap[i] > 0])
                if len(kolona) > 1:
                    kolonski_penali.append(kolona[1] - kolona[0])
                else:
                    kolonski_penali.append(kolona[0] if len(kolona) > 0 else -INF)

        max_red_penal = max(redovni_penali)
        max_kol_penal = max(kolonski_penali)

        if max_red_penal > max_kol_penal:
            i = redovni_penali.index(max_red_penal)
            j = np.argmin([costs[i][j] if pot[j] > 0 else INF for j in range(len(pot))])
        else:
            j = kolonski_penali.index(max_kol_penal)
            i = np.argmin([costs[i][j] if kap[i] > 0 else INF for i in range(len(kap))])

        allocation = min(kap[i], pot[j])
        resenje[i][j] = allocation
        kap[i] -= allocation
        pot[j] -= allocation

        if kap[i] == 0:
            for k in range(len(pot)):
                costs[i][k] = INF
        if pot[j] == 0:
            for k in range(len(kap)):
                costs[k][j] = INF

    return resenje

In [6]:
resenje = severozapadni_ugao(kapacitet, potražnja)
resenje_df = pd.DataFrame(resenje)

print("Inicijalno rešenje (Severozapadni ugao):")
print(resenje_df)

#resenje = metod_najmanjih_cena(kapacitet, potražnja, cene)
#resenje_df = pd.DataFrame(resenje)

#print("Inicijalno rešenje (Metod najmanjih cena):")
#print(resenje_df)

#resenje = fogelov_metod(kapacitet, potražnja, cene)
#resenje_df = pd.DataFrame(resenje, columns=['D1', 'D2', 'D3'], index=['F1', 'F2', 'F3', 'F4'])

#print("Inicijalno rešenje (Fogelov metod):")
#print(resenje_df)

Inicijalno rešenje (Severozapadni ugao):
     0    1    2    3
0  2.0  0.0  0.0  0.0
1  1.0  3.0  2.0  0.0
2  0.0  0.0  2.0  5.0


In [8]:
def iterativni_postupak(cene: np.ndarray, inicijalno_resenje: pd.DataFrame):
    u = {idx: None for idx in inicijalno_resenje.index}
    v = {col: None for col in inicijalno_resenje.columns}
    c_prim = np.zeros_like(cene)
    i_min = None
    j_min = None
    pozitivne_indeksi = []
    # Stavlja nulu u 'u' za najpopunjeniji red inicijalnog resenja (sto manje praznih polja)
    najpopunjeniji_red = inicijalno_resenje.sum(axis=1).idxmax()
    u[najpopunjeniji_red] = 0

    # c[i][j] == u[i] + v[j]
    def korak1():
        print('korak1\n')
        nonlocal u, v
        while True:
            promena = False
            for i_idx in inicijalno_resenje.index:
                for j_col in inicijalno_resenje.columns:
                    i = inicijalno_resenje.index.get_loc(i_idx)
                    j = inicijalno_resenje.columns.get_loc(j_col)
                    if inicijalno_resenje.loc[i_idx, j_col] > 0:
                        if u[i_idx] is not None and v[j_col] is not None:
                            if cene[i, j] != u[i_idx] + v[j_col]:
                                raise ValueError("Neispravna vrednost u[i] + v[j]")
                            continue
                        elif u[i_idx] is None and v[j_col] is not None:
                            u[i_idx] = cene[i, j] - v[j_col]
                            promena = True
                        elif v[j_col] is None and u[i_idx] is not None:
                            v[j_col] = cene[i, j] - u[i_idx]
                            promena = True
            if not promena:
                break
    def korak2():
        print('korak2\n')
        nonlocal c_prim, j_min, i_min, pozitivne_indeksi
        pozitivne_indeksi.clear()
        j_min = None
        i_min = None
        c_prim = np.zeros_like(cene)
        for i_idx in inicijalno_resenje.index:
            for j_col in inicijalno_resenje.columns:
                i = inicijalno_resenje.index.get_loc(i_idx)
                j = inicijalno_resenje.columns.get_loc(j_col)
                if inicijalno_resenje.loc[i_idx, j_col] == 0:
                    if u[i_idx] is not None and v[j_col] is not None:
                        c_prim[i, j] = cene[i, j] - u[i_idx] - v[j_col]
                        if c_prim[i, j] < 0 and (i_min is None or j_min is None or c_prim[i, j] < c_prim[i_min, j_min]):
                            i_min = i
                            j_min = j
                elif inicijalno_resenje.loc[i_idx, j_col] > 0:
                    pozitivne_indeksi.append((i, j))
        return i_min is not None and j_min is not None
    
    def korak3():
        print('korak3\n')
        nonlocal inicijalno_resenje
        put = []
        def rekurzivna_pretraga(i, j, put, pozitivne_indeksi, is_row): #trazi put od i_min, j_min do i_min, j_min
            print('rekurzivna_pretraga\n')
            nonlocal inicijalno_resenje
            if i == i_min or j == j_min and put.len() > 2:
                return put
            if pozitivne_indeksi.len() == 0:
                raise ValueError("Nema više pozitivnih indeksa za pretragu puta")
            for i_sl,j_sl in pozitivne_indeksi:
                if (i_sl, j_sl) not in put:
                    if is_row == True and i_sl == i and j_sl != j:
                        put.append((i_sl, j_sl))
                        pozitivne_indeksi.remove((i_sl, j_sl))
                        rekurzivna_pretraga(i_sl, j_sl, put,pozitivne_indeksi, False)
                    elif is_row == False and j_sl == j and i_sl != i:
                        pozitivne_indeksi.remove((i_sl, j_sl))
                        put.append((i_sl, j_sl))
                        rekurzivna_pretraga(i_sl, j_sl, put,pozitivne_indeksi, True)
                    elif is_row == None and (i_sl == i or j_sl == j):
                        pozitivne_indeksi.remove((i_sl, j_sl))
                        put.append((i_sl, j_sl))
                        rekurzivna_pretraga(i_sl, j_sl, put, pozitivne_indeksi, True)
        
            def novo_resenje(put):
                print('novo_resenje\n')
                nonlocal inicijalno_resenje
                min_oznaka = min(inicijalno_resenje.loc[i, j] for i, j in put)
                for i, j in put:
                    inicijalno_resenje.loc[i, j] += -min_oznaka if put.index((i,j)) % 2 == 0 else min_oznaka
                return inicijalno_resenje

            put = rekurzivna_pretraga(i_min, j_min, put, pozitivne_indeksi, None) 
            novo_resenje(put)
        

    while True:
        korak1()
        if not korak2():
            break
        korak3()
    return inicijalno_resenje

tabela_resenje = iterativni_postupak(cene, resenje_df)
print("Optimalno rešenje:")
print(tabela_resenje)

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2

korak3

korak1

korak2



KeyboardInterrupt: 